# ToysAD-8K DS2 Confirmation and Multi-view Evaluation

**Notebook file:** `08c-toysad8k-ds2-confirmation-and-multiview.ipynb`

This evaluator-only notebook opens the protected DS2 labels after the LocateAnything, SAM 2.1, and DINOv2-small prediction files are complete and hash locked. It evaluates the fixed DS2 operational route v2, including proposal generation failures and masks that are too small for DINO scoring. It measures proposal and mask quality, compares B0 with P1 across supplied, assisted, and automatic routes, and reports fixed camera-spread view subsets and fusion rules.

It does not load or run a model. It does not tune from calibration or internal-lock data.

## Predeclared evaluation

- View counts: 1, 2, 3, 5, 10, and 20.
- Application condition: three evenly spaced views.
- Fusion reports: mean normalized rank and visibility-weighted normalized rank.
- One-anomaly metrics: Top-1, MRR, and scene AP.
- Two-anomaly metrics: AP, Recall@2, and exact-set recovery.
- Uncertainty: paired confidence intervals clustered by base asset.
- Failure analysis: category, anomaly family, object count, anomaly count, rank disagreement, and low-margin scenes.

DS2 is a confirmation cohort. Both fusion reports are retained, but the predeclared mean-rank rule remains the application setting. No accuracy cutoff or DS2-based model tuning is applied.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

sys.dont_write_bytecode = True

from IPython.display import Image as NotebookImage, display

NOTEBOOK_NAME = "08c-toysad8k-ds2-confirmation-and-multiview.ipynb"
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/temp/toysad8k_ds2_confirmation_evaluation" if EXECUTION_ENVIRONMENT == "kaggle" else ".local/runs/toysad8k-ds2-confirmation-evaluation",
)).expanduser().resolve()

In [ ]:
def kaggle_dataset_roots() -> list[Path]:
    input_root = Path("/kaggle/input")
    if not input_root.is_dir():
        return []
    candidates: list[Path] = []
    for pattern in (
        "*",
        "datasets/*/*",
        "datasets/*/*/versions/*",
        "notebooks/*/*",
        "notebooks/*/*/versions/*",
    ):
        candidates.extend(path for path in input_root.glob(pattern) if path.is_dir())
    return sorted(set(candidates), key=lambda path: path.as_posix())


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents))
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for mounted in kaggle_dataset_roots():
            candidates.extend((mounted, mounted / "code"))
    for candidate in candidates:
        if (candidate / "src/visual_peer_inspector/data/cohort_contract.py").is_file():
            return candidate.resolve()
    mounted_names = sorted(path.name for path in input_root.iterdir()) if input_root.is_dir() else []
    raise FileNotFoundError(
        f"Could not locate the Visual Peer Inspector code snapshot. Mounted inputs: {mounted_names}"
    )


def find_package(package_kind: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for mounted in kaggle_dataset_roots():
            candidates.append(mounted)
            candidates.extend(
                child for child in sorted(mounted.iterdir()) if child.is_dir()
            )
    for candidate in candidates:
        record_path = candidate / "package_record.json"
        if not record_path.is_file():
            continue
        try:
            record = json.loads(record_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        is_rgb_package = (
            package_kind == "toysad8k-model-input"
            and record.get("cohort_id") == "toysad8k-development-confirm-300-v1"
            and (candidate / "neutral_corpus_manifest.jsonl").is_file()
            and (candidate / "media").is_dir()
        )
        if record.get("package_kind") == package_kind or is_rgb_package:
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate package kind: {package_kind}")


def find_named_input_file(name: str, variable: str) -> Path:
    configured = os.getenv(variable)
    if configured:
        path = Path(configured).expanduser()
        if path.is_file():
            return path.resolve()
    patterns = (
        name,
        f"*/{name}",
        f"*/*/{name}",
        f"*/*/*/{name}",
        f"*/*/*/*/{name}",
    )
    matches = []
    for mounted in kaggle_dataset_roots():
        for pattern in patterns:
            matches.extend(path for path in mounted.glob(pattern) if path.is_file())
    unique = sorted(set(matches), key=lambda path: path.as_posix())
    if not unique:
        raise FileNotFoundError(f"Could not find mounted input file: {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in unique}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} do not match")
    preferred = sorted(
        unique,
        key=lambda path: ("downstream" not in path.parts, path.as_posix()),
    )
    return preferred[0].resolve()


def read_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source_root = str(CODE_ROOT / "src")
    previous = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source_root if not previous else source_root + os.pathsep + previous
    return environment


def run_clearml_finalizer(safe_root: Path, stage: str, task_name: str, tags: list[str], receipt: Path) -> dict:
    mode = "online" if EXECUTION_ENVIRONMENT == "kaggle" else "disabled"
    environment = code_environment()
    if mode == "online":
        access = read_secret("CLEARML_API_ACCESS_KEY")
        secret = read_secret("CLEARML_API_SECRET_KEY")
        if not access or not secret:
            mode = "disabled"
        else:
            environment.update({
                "CLEARML_API_ACCESS_KEY": access,
                "CLEARML_API_SECRET_KEY": secret,
                "CLEARML_API_HOST": "https://api.clear.ml",
                "CLEARML_WEB_HOST": "https://app.clear.ml",
                "CLEARML_FILES_HOST": "https://files.clear.ml",
            })
    command = [
        sys.executable,
        str(CODE_ROOT / "scripts/finalize_ds2_clearml.py"),
        "--safe-root", str(safe_root),
        "--receipt", str(receipt),
        "--stage", stage,
        "--task-name", task_name,
        "--mode", mode,
    ]
    for tag in tags:
        command.extend(("--tag", tag))
    try:
        subprocess.run(command, env=environment, check=True, timeout=240)
    except Exception as error:
        receipt.parent.mkdir(parents=True, exist_ok=True)
        receipt.write_text(json.dumps({
            "schema_version": 1,
            "status": "failed",
            "error_type": type(error).__name__,
            "task_id": None,
        }, indent=2) + "\\n", encoding="utf-8")
    return json.loads(receipt.read_text(encoding="utf-8"))

## Locate the frozen predictions and protected evaluator inputs

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
RGB_ROOT = find_package("toysad8k-model-input", "VPI_RGB_ROOT")
GEOMETRY_ROOT = find_package("ds2-oracle-geometry", "VPI_GEOMETRY_ROOT")
ATTRIBUTES_ROOT = find_package("ds2-evaluator-attributes", "VPI_ATTRIBUTES_ROOT")


def verified_extract(archive_path: Path, destination: Path) -> None:
    # Validate archive integrity and member paths before evaluator inputs are opened.
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError(f"Archive integrity failed: {archive_path.name}")
        for name in archive.namelist():
            member = Path(name)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Unsafe archive member: {name}")
        archive.extractall(destination)


def verified_downstream_archive(archive_name: str) -> Path:
    stage_name = archive_name.removesuffix("-downstream-input.zip")
    archive_path = find_named_input_file(archive_name, "VPI_UNUSED_ARCHIVE")
    verification_path = find_named_input_file(
        f"{stage_name}-verification.json",
        "VPI_UNUSED_VERIFICATION",
    )
    verification = json.loads(verification_path.read_text(encoding="utf-8"))
    matches = [row for row in verification["archives"] if row["path"] == archive_name]
    if len(matches) != 1:
        raise ValueError(f"No unique verification record for {archive_name}")
    actual_sha256 = sha256(archive_path.read_bytes()).hexdigest()
    if actual_sha256 != matches[0]["sha256"]:
        raise ValueError(f"Archive SHA-256 differs from the producer record: {archive_name}")
    checksums_path = verification_path.parent / "SHA256SUMS"
    if not checksums_path.is_file():
        raise FileNotFoundError(f"Checksum manifest is missing for {archive_name}")
    checksums = {
        line.split(maxsplit=1)[1].strip(): line.split(maxsplit=1)[0]
        for line in checksums_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }
    if checksums.get(archive_name) != actual_sha256:
        raise ValueError(f"Archive SHA-256 differs from SHA256SUMS: {archive_name}")
    return archive_path


configured_inference = os.getenv("VPI_INFERENCE_ROOT")
if configured_inference:
    INFERENCE_ROOT = Path(configured_inference).expanduser().resolve()
else:
    INFERENCE_ROOT = Path("/kaggle/temp/ds2_evaluator_inference_inputs").resolve()
    marker = INFERENCE_ROOT / "representation/model-input/package_record.json"
    if not marker.is_file():
        archive_specs = [
            ("ds2-reference-sam21-downstream-input.zip", INFERENCE_ROOT / "sam21/reference-box"),
            ("ds2-automatic-sam21-downstream-input.zip", INFERENCE_ROOT / "sam21/automatic-box"),
            ("ds2-dinov2-p1-downstream-input.zip", INFERENCE_ROOT / "representation"),
        ]
        for archive_name, destination in archive_specs:
            destination.mkdir(parents=True, exist_ok=True)
            verified_extract(verified_downstream_archive(archive_name), destination)

SCORE_ROOT = INFERENCE_ROOT / "representation/model-input"
REFERENCE_SAM_RESULTS = INFERENCE_ROOT / "sam21/reference-box/private/sam_results.jsonl"
AUTOMATIC_SAM_RESULTS = INFERENCE_ROOT / "sam21/automatic-box/private/sam_results.jsonl"
for required in (SCORE_ROOT / "package_record.json", REFERENCE_SAM_RESULTS, AUTOMATIC_SAM_RESULTS):
    if not required.is_file():
        raise FileNotFoundError(f"Frozen DS2 evaluator input is missing: {required.name}")

score_record = json.loads((SCORE_ROOT / "package_record.json").read_text(encoding="utf-8"))
display({
    "score package status": score_record["all_checks_passed"],
    "score rows": score_record["score_row_count"],
    "model": score_record["model_key"],
    "methods": score_record["methods"],
    "evaluator opened during inference": score_record["evaluator_opened"],
    "reference SAM results": REFERENCE_SAM_RESULTS.name,
    "automatic SAM results": AUTOMATIC_SAM_RESULTS.name,
})

## Run the protected confirmation and multi-view analysis

In [ ]:
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_ds2_confirmation_evaluation.py"),
    "--rgb-root", str(RGB_ROOT),
    "--geometry-root", str(GEOMETRY_ROOT),
    "--attributes-root", str(ATTRIBUTES_ROOT),
    "--inference-root", str(INFERENCE_ROOT),
    "--output-root", str(OUTPUT_ROOT),
]
subprocess.run(command, check=True, env=code_environment())
run_record = json.loads((OUTPUT_ROOT / "safe/run_record.json").read_text(encoding="utf-8"))
display({
    "status": run_record["status"],
    "scenes": run_record["scene_count"],
    "metric rows": run_record["scene_metric_row_count"],
    "unscoreable candidate-view rows": run_record["unscoreable_candidate_view_count"],
    "invalid-mask candidate-view rows": run_record["invalid_mask_candidate_view_count"],
    "prediction lock verified": run_record["evaluator_opened_after_prediction_lock"],
})

## Component confirmation

In [ ]:
import pandas as pd

SAFE_ROOT = OUTPUT_ROOT / "safe"
display(pd.read_csv(SAFE_ROOT / "proposal_summary.csv"))
display(pd.read_csv(SAFE_ROOT / "mask_summary.csv"))
display(pd.read_csv(SAFE_ROOT / "unscoreable_summary.csv"))
display(pd.read_csv(SAFE_ROOT / "decision_gates.csv"))
display(pd.read_csv(SAFE_ROOT / "paired_intervals.csv"))
display(NotebookImage(filename=str(SAFE_ROOT / "locateanything_proposal_quality.png")))
display(NotebookImage(filename=str(SAFE_ROOT / "sam_mask_quality.png")))
display(NotebookImage(filename=str(SAFE_ROOT / "sam_mask_iou_distribution.png")))
display(NotebookImage(filename=str(SAFE_ROOT / "route_average_precision.png")))

## View-count and fusion results

In [ ]:
multiview = pd.read_csv(SAFE_ROOT / "multiview_summary.csv")
automatic_curve = multiview.query("route == 'automatic_box' and method == 'P1'")
display(automatic_curve.sort_values(["view_count", "fusion_method"]))
display(multiview.query("view_count == 20 and fusion_method == 'mean_rank'").sort_values(["route", "method"]))
display(NotebookImage(filename=str(SAFE_ROOT / "multiview_ablation.png")))
display(pd.read_csv(SAFE_ROOT / "primary_three_view_summary.csv"))
display(NotebookImage(filename=str(SAFE_ROOT / "primary_three_view_metrics.png")))
display(NotebookImage(filename=str(SAFE_ROOT / "rank_disagreement_distribution.png")))
print("Fusion uses only scored ranks and applies scored-view support once. Detected support remains a separate coverage measure.")

## Subgroups and real failure examples

In [ ]:
subgroups = pd.read_csv(SAFE_ROOT / "subgroup_summary.csv")
display(subgroups.sort_values(["dimension", "average_precision"]).groupby("dimension").head(8))
display(pd.read_csv(SAFE_ROOT / "scene_type_summary.csv"))
display(NotebookImage(filename=str(SAFE_ROOT / "scene_type_metrics.png")))
display(NotebookImage(filename=str(SAFE_ROOT / "lowest_supported_subgroups.png")))
print("The private panel labels each example by route and score. Box and mask outlines show the evidence used for each prediction, and missed references are marked explicitly.")
display(NotebookImage(filename=str(OUTPUT_ROOT / "private/figures/three_view_success_and_failure.png")))

## Frozen configuration and confirmation status

In [ ]:
decision = json.loads((SAFE_ROOT / "model_and_route_decision.json").read_text(encoding="utf-8"))
display(pd.DataFrame([decision]))
print(f"Encoder: {decision['encoder']}")
print(f"Scorer: {decision['scorer']}")
print(f"Application mask route: {decision['application_mask_route']}")
print(f"Primary views: {decision['primary_view_ids']}")
print(f"Fusion: {decision['fusion_method']}")
print(f"Decision status: {decision['status']}")

## Bounded ClearML finalisation

Only aggregate tables, figures, decisions, and hashes are uploaded. Per-scene scores, object IDs, labels, masks, RGB images, and failure panels remain in private Kaggle output.

In [ ]:
if EXECUTION_ENVIRONMENT == "kaggle":
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")], check=True, timeout=180)
    except Exception as error:
        print(f"ClearML dependency setup was unavailable: {type(error).__name__}")
tracking_receipt = run_clearml_finalizer(
    SAFE_ROOT,
    "ds2_confirmation_and_multiview",
    "ds2-confirmation-and-multiview",
    ["notebook-08c", "ds2", "multiview", "fusion", "kaggle"],
    OUTPUT_ROOT / "clearml_receipt.json",
)
display(tracking_receipt)

## Build compact verified downloads

The repository archive contains aggregate tables, charts, decisions, and hashes. The review archive contains only the selected private success and failure panel. Per-scene scores, masks, RGB images, credentials, and extracted evaluator inputs are not included in the GitHub candidate.

In [ ]:
BUNDLE_ROOT = (
    Path("/kaggle/working/notebook08c_bundles")
    if EXECUTION_ENVIRONMENT == "kaggle"
    else OUTPUT_ROOT.parent / "notebook08c_bundles"
)
verification_path = BUNDLE_ROOT / "ds2-confirmation-and-multiview-verification.json"
if not verification_path.is_file():
    subprocess.run(
        [
            sys.executable,
            str(CODE_ROOT / "scripts/package_kaggle_stage_output.py"),
            "--stage-root", str(OUTPUT_ROOT),
            "--output-root", str(BUNDLE_ROOT),
            "--stage-name", "ds2-confirmation-and-multiview",
            "--review-path", "private/figures",
            "--review-path", "clearml_receipt.json",
        ],
        check=True,
        env=code_environment(),
    )
bundle_record = json.loads(verification_path.read_text(encoding="utf-8"))
display(pd.DataFrame(bundle_record["archives"]))
print(f"Verified downloads: {BUNDLE_ROOT}")

## Next stage

Record the measured automatic and assisted routes separately. DS2 measures the predeclared SAM 2.1 Hiera-large, DINOv2-small, B0, P1, three-view, and mean-rank configuration without tuning it. Keep that configuration unchanged for the next development scale run. The automatic route remains experimental until a label-free cross-view association method replaces the evaluator-only box matching used in this notebook.

The next two weeks should scale the frozen method to the 1,000-scene development role, implement label-free cross-view association and the assisted review flow, then use the 600-scene calibration role only to set a review threshold. Internal-lock and official evaluation remain unopened until those choices are frozen.